# Man-in-the-middle and ARP spoofing

# Coding problem, 3.1.c: ARP spoofing detector

**Skill:** Analyze Risk · **Maps to:** 3.1.B · **Language:** Python 3 (pytest)

## Problem
You are building an arpwatch-style detector for Crestfield Analytics. Given a timestamp-ordered
log of observed ARP replies (each mapping an IP to a MAC), identify which IP addresses show the
ARP spoofing signature: the **first** reply seen for an IP establishes its trusted MAC binding. Any
**later** reply for the same IP with a **different** MAC is a spoofing indicator. A later reply
with the *same* MAC as the trusted binding is just a normal refresh and is not flagged.

## Function contract
Implement `solve(replies)` in the code editor (see the Reference section below).

## Input
- `replies`: `list[dict]`, each with:
  - `"ts"`: `int`, Unix timestamp (seconds) the reply was observed
  - `"ip"`: `str`, the IP address claimed in the reply
  - `"mac"`: `str`, the MAC address claimed for that IP

## Output
A `list[str]`, the distinct IP addresses that show the spoofing signature (at least one reply with
a MAC different from that IP's first-seen MAC), **sorted alphabetically**, with no duplicates.

## Examples
```python
replies = [
    {"ts": 100, "ip": "192.168.1.1", "mac": "aa:11:22:33:44:01"},
    {"ts": 130, "ip": "192.168.1.1", "mac": "aa:11:22:33:44:01"},
    {"ts": 145, "ip": "192.168.1.1", "mac": "de:ad:be:ef:00:99"},
]
solve(replies)  ->  ["192.168.1.1"]

replies2 = [
    {"ts": 100, "ip": "192.168.1.50", "mac": "11:22:33:44:55:66"},
]
solve(replies2)  ->  []
```

## Constraints
- Deterministic, offline, standard library only.
- `replies` may be given in any timestamp order in the list; use the `ts` field (not list
  position) to determine which reply for a given IP came first (the "trusted" binding).
- Test data never has two replies for the same IP with an identical minimum `ts`.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(replies: list[dict]) -> list[str]
```

### Input (`replies`)
`list[dict]` where each dict has:
- `"ts"`: `int`, Unix timestamp in seconds the reply was observed
- `"ip"`: `str`, IP address claimed in the reply
- `"mac"`: `str`, MAC address claimed for that IP

### Output (`result`)
A `list[str]` of distinct IP addresses that show the ARP spoofing signature, **sorted
alphabetically**, with no duplicate entries.

### Detection rule
1. For each IP, find the reply with the smallest `ts`, this establishes the trusted MAC binding
   for that IP.
2. If any other reply for that same IP has a `mac` different from the trusted binding, the IP is
   flagged (spoofed).
3. IPs with only one observed reply, or where every reply agrees on the same MAC, are not flagged.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- `replies` may arrive in any order; use `ts` (not list position) to determine the trusted (first)
  binding.
- Test data never has two replies for the same IP with a tied minimum `ts`.
- Output is sorted alphabetically with no duplicates.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Your implementation
"""Starter code for 3.1.c. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""
from typing import List, Dict


def solve(replies: List[Dict]) -> List[str]:
    """Flag IP addresses that show the ARP spoofing signature.

    Args:
        replies: list of dicts with keys 'ts' (int), 'ip' (str), 'mac' (str).
    Returns:
        Sorted list (no duplicates) of IP addresses where a later reply's MAC differs from
        the MAC of the earliest (by 'ts') reply for that IP.
        See the problem statement above and the Reference section below for the exact rules.
    """
    # TODO: implement the detection rule per the problem statement above
    raise NotImplementedError("implement solve() — see the problem statement above")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

def _r(ts, ip, mac):
    return {"ts": ts, "ip": ip, "mac": mac}


# ---- visible cases -------------------------------------------------------

def test_no_spoof_same_mac_refresh():
    replies = [
        _r(0, "192.168.1.1", "aa:01"),
        _r(30, "192.168.1.1", "aa:01"),
    ]
    assert impl.solve(replies) == []


def test_single_reply_not_flagged():
    replies = [_r(0, "192.168.1.50", "11:22")]
    assert impl.solve(replies) == []


def test_spoof_different_mac_flagged():
    replies = [
        _r(0, "192.168.1.1", "aa:01"),
        _r(15, "192.168.1.1", "de:ad"),
    ]
    assert impl.solve(replies) == ["192.168.1.1"]


def test_multiple_ips_only_spoofed_flagged():
    replies = [
        _r(0, "192.168.1.1", "aa:01"),
        _r(15, "192.168.1.1", "de:ad"),
        _r(0, "192.168.1.2", "bb:02"),
        _r(60, "192.168.1.2", "bb:02"),
    ]
    assert impl.solve(replies) == ["192.168.1.1"]


def test_sorted_output_no_duplicates():
    replies = [
        _r(0, "192.168.1.9", "aa:01"),
        _r(5, "192.168.1.9", "bb:02"),
        _r(10, "192.168.1.9", "cc:03"),
        _r(0, "192.168.1.2", "ff:00"),
        _r(5, "192.168.1.2", "ee:01"),
    ]
    assert impl.solve(replies) == ["192.168.1.2", "192.168.1.9"]


def test_empty_replies():
    assert impl.solve([]) == []


# ---- hidden / edge cases -------------------------------------------------

def test_replies_out_of_ts_order_in_list():
    """List order is not sorted by ts; the 'ts' field, not position, determines trust."""
    replies = [
        _r(50, "10.0.0.5", "attacker-mac"),
        _r(10, "10.0.0.5", "trusted-mac"),
    ]
    assert impl.solve(replies) == ["10.0.0.5"]


def test_three_replies_same_mac_after_first_change():
    """Once flagged (differs from trusted), further same-mac repeats don't un-flag it."""
    replies = [
        _r(0, "10.0.0.9", "trusted"),
        _r(5, "10.0.0.9", "attacker"),
        _r(10, "10.0.0.9", "attacker"),
    ]
    assert impl.solve(replies) == ["10.0.0.9"]


def test_stable_binding_many_refreshes():
    replies = [_r(t, "10.0.0.20", "stable-mac") for t in range(0, 100, 10)]
    assert impl.solve(replies) == []


def test_large_mixed_batch():
    replies = []
    expected = []
    for i in range(10):
        ip = f"10.0.1.{i}"
        replies.append(_r(0, ip, "m0"))
        if i % 2 == 0:
            replies.append(_r(10, ip, "m1"))  # different -> spoofed
            expected.append(ip)
        else:
            replies.append(_r(10, ip, "m0"))  # same -> not spoofed
    assert impl.solve(replies) == sorted(expected)


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
